In [4]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import openai
import pandas as pd

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

# === Configuration ===
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
COLLECTION_NAME = "test_collection_oai"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

#### Initiate Qdrant client

In [13]:
# qdrant_client = QdrantClient(url="http://localhost:6333")

# qdrant_client.create_collection(
#     collection_name="Amazon-items-collection-01",
#     vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
# )


qdrant_client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY  # Add this line
)

#### Read the sampled dataset with Amazon inventory metadata

In [ ]:
df_items = pd.read_json("../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

#### Concatenate title and features

In [ ]:
def preprocess_data(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [ ]:
df_items["preprocessed_data"] = df_items.apply(preprocess_data, axis=1)

#### Sample 50 items from the dataset

In [ ]:
df_sample = df_items.sample(n=50, random_state=42)

#### Define the embeddings function

In [8]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=[text],
        model=model,
    )
    return response.data[0].embedding

#### Embed data

In [ ]:
data_to_embed = df_sample["preprocessed_data"].tolist()
pointstructs = []
for i, data in enumerate(data_to_embed):
    embedding = get_embedding(data)
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload={"text": data},
        )
    )

#### Write embedded data to Qdrant

In [ ]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-00",
    wait=True,
    points=pointstructs,
)

#### Define a function for data retrieval

In [21]:
# def retrieve_data(query):
#     query_embedding = get_embedding(query)
#     results = qdrant_client.query_points(
#         collection_name="Amazon-items-collection-00",
#         query=query_embedding,
#         limit=10,
#     )
#     return results

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=[text],
        model=model,
    )
    return response.data[0].embedding

def retrieve_data(query):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="test_collection_oai",
        query=query_embedding,
        limit=10,
    )
    return results






In [22]:
from qdrant_client.models import Prefetch, Filter, FieldCondition, MatchText, FusionQuery

def retrieve_data_hybrid(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="test_collection_oai",
        prefetch=[
            Prefetch(
                query=query_embedding,
                limit=20
            ),
            Prefetch(
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="text",
                            match=MatchText(text=query)
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []

    for result in results.points:
        retrieved_context_ids.append(result.id)
        retrieved_context.append(result.payload['text'])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores
    }

#### Test data retrieval

In [23]:
# retrieve_data("What airphones can I get?").points
retrieve_data("What do we know about star clusters in the M33 galaxy?").points


[ScoredPoint(id='16b91995-3fa1-4b4c-a7e6-7ddfdaf1f35c', version=21, score=0.7603514, payload={'file_name': 'Thesis_de_Meulenaer.pdf', 'file_hash': 'd57f0a140ab6b5d0c0d656f39f947ca4e6168fc4ab5732b22cc10f91fcd0ad9e', 'file_title': '', 'authors': '', 'keywords': '', 'creation_date': "D:20150818115209+03'00'", 'page_number': '79', 'text': '3.3\nApplication to the M33 star clusters\n3.3.1\nWhy M33?\nThere is a current need for an accurate catalog for the star cluster system\nof the Triangulum galaxy, or Messier 33 (M33), as it could be used as a\nconstraint on the derivation of star formation history in this galaxy. Several\nother reasons encourage the study of this particular star cluster system. The\nnearly face-on inclination (i = 56 degrees, Regan & Vogel 1994) of M33\nreduces extinction eﬀects for the majority of its cluster population, situated\nin the disk. Also, M33 is the only close late-type spiral galaxy, situated at\na distance of 867 kpc (Galleti et al. 2004, distance modulus o

In [24]:
retrieve_data_hybrid("What do we know about star clusters in the M33 galaxy?")

{'retrieved_context_ids': ['16b91995-3fa1-4b4c-a7e6-7ddfdaf1f35c',
  '1e9780f7-dc1f-42bf-9493-4fe0afe96ef3',
  '0fcbaaf8-7e48-4ddf-b2d0-ea86205094ff',
  'f28b9c8c-69be-4e09-a701-b35eb292daa2',
  'ddf9662b-2024-445c-af08-0a1946629d89'],
 'retrieved_context': ['3.3\nApplication to the M33 star clusters\n3.3.1\nWhy M33?\nThere is a current need for an accurate catalog for the star cluster system\nof the Triangulum galaxy, or Messier 33 (M33), as it could be used as a\nconstraint on the derivation of star formation history in this galaxy. Several\nother reasons encourage the study of this particular star cluster system. The\nnearly face-on inclination (i = 56 degrees, Regan & Vogel 1994) of M33\nreduces extinction eﬀects for the majority of its cluster population, situated\nin the disk. Also, M33 is the only close late-type spiral galaxy, situated at\na distance of 867 kpc (Galleti et al. 2004, distance modulus of (m −M)0 =\n24.69), making its star cluster system accessible to ground-based

In [7]:
import openai
from langsmith.wrappers import wrap_openai
from langsmith import traceable

# Auto-trace LLM calls in-context
client = wrap_openai(openai.Client())

@traceable # Auto-trace this function
def pipeline(user_input: str):
    result = client.chat.completions.create(
        messages=[{"role": "user", "content": user_input}],
        model="gpt-3.5-turbo"
    )
    return result.choices[0].message.content

pipeline("Hello, world!")
# Out:  Hello there! How can I assist you today?

'Hello! How can I assist you today?'